# WoundMap AI: Phase 3 — Preprocessing & Exploratory Data Analysis (EDA)

**Project**: WoundMap AI (Biophysics & Biomedical Computer-Vision Research Prototype)  
**Governing Blueprint**: `WoundMapAI_Antigravity_Autonomous_Execution_Blueprint.pdf`  
**Objective**: Provide an end-to-end, thoroughly documented implementation of all preprocessing, color transformations, illumination normalisation routines, and paired geometric augmentations.

---

## Table of Contents
1. **Process 1**: Environment Configuration & Reproducibility Controls
2. **Process 2**: Manifest Ingestion & Stratified Split Inspection
3. **Process 3**: Discrete Mask Decoding (RGBA to Categorical Indices)
4. **Process 4**: Paired Resizing: Continuous Bicubic Image vs Discrete Nearest-Neighbour Mask
5. **Process 5**: Mathematical Color Space Transformations (sRGB, HSV, CIELAB)
6. **Process 6**: Contrast-Limited Adaptive Histogram Equalization (CLAHE) on Luminance
7. **Process 7**: Synchronous Geometric Augmentation with Landmark Alignment Verification
8. **Process 8**: PyTorch Dataset & Batch Tensor Generation

### Process 1: Environment Configuration & Reproducibility Controls
We lock the random seed across Python, NumPy, and PyTorch to guarantee identical data ordering, transform sequences, and batch collation across all runs.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Reproducibility seed set to {SEED}. PyTorch version: {torch.__version__}")

### Process 2: Manifest Ingestion & Stratified Split Inspection
We load the locked train, validation, and test split manifests generated during Phase 2. As established in the protocol, the test partition is strictly held out and remains unaccessed during model training.

In [ ]:
train_df = pd.read_csv(os.path.join("..", "data", "manifests", "train.csv"))
val_df = pd.read_csv(os.path.join("..", "data", "manifests", "val.csv"))
test_df = pd.read_csv(os.path.join("..", "data", "manifests", "test.csv"))

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}")
train_df[["image_id", "height", "width", "classes_present", "pct_granulation", "pct_slough"]].head()

### Process 3: Discrete Mask Decoding (RGBA to Categorical Indices)
Raw WoundTissue masks are 4-channel RGBA images. To train a semantic segmentation model, each RGB triplet must be mapped to a discrete integer class index $c \in \{0, 1, 2, 3, 4, 5, 6\}$ according to the audited palette:
- 0: Background (`(0, 0, 255)` — Blue)
- 1: Granulation (`(255, 0, 0)` — Red)
- 2: Slough (`(255, 255, 0)` — Yellow)
- 3: Necrosis (`(0, 0, 0)` — Black)
- 4: Maceration (`(255, 255, 255)` — White)
- 5: Bone (`(255, 200, 124)` — Cream/Peach)
- 6: Tendon (`(0, 191, 255)` — Sky Blue)

In [ ]:
from src.woundmap.masks import rgb_to_mask, mask_to_rgb, WOUNDTISSUE_PALETTE

sample_row = train_df.iloc[0]
with Image.open(os.path.join("..", sample_row["mask_path"])) as raw_mask:
    mask_arr = np.array(raw_mask)

discrete_mask = rgb_to_mask(mask_arr, palette=WOUNDTISSUE_PALETTE)
print(f"Raw mask shape: {mask_arr.shape} -> Discrete mask shape: {discrete_mask.shape}")
print(f"Unique classes present in {sample_row['image_id']}: {np.unique(discrete_mask)}")

### Process 4: Paired Resizing: Continuous Bicubic Image vs Discrete Nearest-Neighbour Mask
> **Critical Medical Safeguard**:
> - Photographic images represent continuous optical intensity and are resized using **Bicubic interpolation** for smooth feature representation.
> - Segmentation masks represent discrete categorical labels. Applying linear, bilinear, or bicubic interpolation to masks causes label bleeding and creates spurious intermediate class IDs (e.g. interpolating between class 0 [Background] and class 2 [Slough] would synthesize non-existent class 1 [Granulation]). Categorical masks **MUST** strictly use **Nearest-Neighbour interpolation**.

In [ ]:
from src.woundmap.preprocessing import paired_resize

with Image.open(os.path.join("..", sample_row["image_path"])) as raw_img,
     Image.open(os.path.join("..", sample_row["mask_path"])) as raw_mask:
    img_512, mask_512 = paired_resize(raw_img.convert("RGB"), raw_mask.convert("RGB"), target_size=(512, 512))

res_discrete = rgb_to_mask(np.array(mask_512), WOUNDTISSUE_PALETTE)
print(f"Resized Image size: {img_512.size} | Resized Mask size: {mask_512.size}")
print(f"Discrete classes post-resize (no spurious classes created): {np.unique(res_discrete)}")
assert set(np.unique(res_discrete)).issubset(set(np.unique(discrete_mask))), "Spurious class detected!"

### Process 5: Mathematical Color Space Transformations (sRGB, HSV, CIELAB)
We investigate whether decoupling chrominance from luminance improves segmentation robustness:
1. **sRGB**: Standard correlated channels scaled to $[0.0, 1.0]$.
2. **HSV**: Cylindrical representation separating Hue ($H \in [0, 1]$) and Saturation ($S \in [0, 1]$) from Value / Light Intensity ($V \in [0, 1]$).
3. **CIELAB**: Perceptually uniform space based on opponent color theory:
   - $L^* \in [0, 100]$: Perceived luminance.
   - $a^* \in [-128, 127]$: Opponent Green-to-Red axis.
   - $b^* \in [-128, 127]$: Opponent Blue-to-Yellow axis.
   - Normalized linearly to $[0.0, 1.0]$ for stable gradient propagation in deep networks.

In [ ]:
from src.woundmap.preprocessing import convert_color_space

img_np = np.array(img_512)
srgb_t = convert_color_space(img_np, "RGB")
hsv_t = convert_color_space(img_np, "HSV")
lab_t = convert_color_space(img_np, "CIELAB")

print(f"sRGB bounds:   [{srgb_t.min():.3f}, {srgb_t.max():.3f}] | dtype: {srgb_t.dtype}")
print(f"HSV bounds:    [{hsv_t.min():.3f}, {hsv_t.max():.3f}] | dtype: {hsv_t.dtype}")
print(f"CIELAB bounds: [{lab_t.min():.3f}, {lab_t.max():.3f}] | dtype: {lab_t.dtype}")

### Process 6: Contrast-Limited Adaptive Histogram Equalization (CLAHE) on Luminance
Standard global histogram equalization over-amplifies noise in shadowed or highlight regions. CLAHE operates over local grid tiles ($8 \times 8$) and clips histogram bins at a contrast threshold $\beta = 2.0$.

In our protocol, CLAHE is applied **strictly to the luminance channel**:
- In CIELAB: Applied to $L^*$ (Lightness), keeping opponent color channels $a^*, b^*$ uncorrupted.
- In HSV: Applied to $V$ (Value), keeping pure hue $H$ and saturation $S$ intact.

In [ ]:
from src.woundmap.preprocessing import preprocess_image

# Apply CLAHE strictly to L* channel in CIELAB
lab_clahe = preprocess_image(img_np, color_space="CIELAB", illumination_norm="clahe", clip_limit=2.0)
# Apply CLAHE strictly to V channel in HSV
hsv_clahe = preprocess_image(img_np, color_space="HSV", illumination_norm="clahe", clip_limit=2.0)

print(f"CIELAB CLAHE-L output shape: {lab_clahe.shape} | bounds: [{lab_clahe.min():.3f}, {lab_clahe.max():.3f}]")
print(f"HSV CLAHE-V output shape:    {hsv_clahe.shape} | bounds: [{hsv_clahe.min():.3f}, {hsv_clahe.max():.3f}]")

### Process 7: Synchronous Geometric Augmentation with Landmark Alignment Verification
> **Scientific Safeguard 6**: Apply geometric transforms identically to image and categorical mask. Color transforms apply to image only.

We verify that horizontal flips, vertical flips, and $90^\circ$ rotations alter image coordinates and mask coordinates in exact lockstep.

In [ ]:
from src.woundmap.preprocessing import paired_geometric_transform

# Set synthetic test landmark at top-left pixel (0, 0)
test_img = np.zeros((64, 64, 3), dtype=np.uint8)
test_mask = np.zeros((64, 64), dtype=np.int64)
test_img[0, 0] = [255, 255, 255]
test_mask[0, 0] = 1

img_h, mask_h = paired_geometric_transform(test_img, test_mask, hflip=True)
print(f"Original landmark: (0, 0) | H-Flipped Image landmark: {np.argwhere(img_h[:, :, 0] == 255)[0]}")
print(f"Original mask:     (0, 0) | H-Flipped Mask landmark:  {np.argwhere(mask_h == 1)[0]}")
assert tuple(np.argwhere(img_h[:, :, 0] == 255)[0]) == tuple(np.argwhere(mask_h == 1)[0]), "Alignment mismatch!"

### Process 8: PyTorch Dataset & Batch Tensor Generation
We instantiate the `WoundDataset` and `build_dataloader` pipeline to verify that training batches are produced in PyTorch tensor format $(B, 3, 512, 512)$ and $(B, 512, 512)$ with correct tensor dtypes and class index boundaries.

In [ ]:
from src.woundmap.data import build_dataloader

loader = build_dataloader(
    manifest_csv=os.path.join("..", "data", "manifests", "train.csv"),
    batch_size=4,
    shuffle=True,
    target_size=(512, 512),
    color_space="CIELAB",
    illumination_norm="clahe",
    augment=True,
    seed=SEED
)

batch = next(iter(loader))
images, masks, ids = batch["image"], batch["mask"], batch["image_id"]

print(f"Batch Image Tensor: shape={images.shape}, dtype={images.dtype}, bounds=[{images.min():.3f}, {images.max():.3f}]")
print(f"Batch Mask Tensor:  shape={masks.shape}, dtype={masks.dtype}, class_range=[{masks.min()}, {masks.max()}]")
print(f"Batch Image IDs:    {ids}")
print("\nPhase 3 Preprocessing & EDA Pipeline successfully verified end-to-end!")